# Entrega 04: auditoria, partição e vazamento

Seção de `Kaio Vitor`. O notebook lê os três CSVs brutos sem alterá-los, registra cada conferência numa tabela final, propõe a partição temporal por data e classifica variáveis quanto a vazamento.

## 0. Parâmetros

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


def achar_raiz():
    atual = Path.cwd().resolve()
    for pasta in [atual, *atual.parents]:
        if (pasta / "dados" / "bruto").is_dir():
            return pasta
    raise FileNotFoundError("Pasta dados/bruto não encontrada. Abra o notebook dentro do projeto.")



# Ponto único de configuração. Para trocar o mock pelos dados reais,
# depois mudar só esta célula: pasta, nomes de arquivo, colunas e valores esperados.
RAIZ = achar_raiz()
PASTA_DADOS = RAIZ / "dados" / "bruto" / "mock"
ARQUIVOS = {
    "catraca": "catraca_ru.csv",
    "estudantes": "estudantes.csv",
    "componentes": "matriculas_componentes.csv",
}
CHAVES = {
    "catraca": ["id_registro_utilizacao"],
    "estudantes": ["id_matricula", "periodo_letivo"],
    "componentes": ["id_matricula_componente"],
}

# Números do relatório do mock. Com dados reais, são os valores a conferir.
LINHAS_ESPERADAS = {"catraca": 224880, "estudantes": 2016, "componentes": 5922}
SERVIDAS_ESPERADAS = 221600
NEGADAS_ESPERADAS = 3280

REFEICOES = ["cafe_da_manha", "almoco", "jantar"]
AUXILIOS = {"residencia_estudantil", "auxilio_alimentacao", "sem_auxilio"}
CATEGORIAS = {
    "catraca": {
        "unidade_ru": {"RU_CAVN_SIMULADO"},
        "tipo_refeicao": set(REFEICOES),
        "tipo_acesso": {"total", "parcial"},
        "tipo_liberacao": {"automatica", "manual"},
        "nivel_ensino": {"tecnico"},
        "tipo_auxilio": AUXILIOS,
        "situacao_evento": {"servida", "negada"},
    },
    "estudantes": {
        "situacao_vinculo_periodo": {"ativo", "trancado", "cancelado", "concluido"},
        "residente": {"True", "False"},
        "tipo_auxilio": AUXILIOS,
        "nivel_ensino": {"tecnico"},
    },
    "componentes": {
        "modalidade": {"presencial", "hibrida"},
        "resultado_final": {"aprovado", "reprovado"},
    },
}
FORMATOS = {
    ("estudantes", "matricula"): r"\d{10}",
    ("estudantes", "id_matricula"): r"MAT_SIM_\d{4}",
    ("estudantes", "id_discente"): r"DIS_SIM_\d{4}",
    ("estudantes", "periodo_letivo"): r"\d{4}\.[12]",
}
CAMPOS_CALENDARIO = [
    "inicio_recesso", "fim_recesso", "inicio_periodo_letivo",
    "fim_periodo_letivo", "datas_suspensao_atividades",
]
DIAS_AULA = {"SEG", "TER", "QUA", "QUI", "SEX"}

# Cortes provisórios da partição. Com dados reais, precisam ser revisados.
FIM_TREINO = pd.Timestamp("2024-12-31")
FIM_VALIDACAO = pd.Timestamp("2025-12-31")

## 1. Leitura e reconciliação de contagens

Os arquivos são lidos como texto (`dtype=str`), com os vazios preservados, e nenhum arquivo bruto é alterado. Cada conferência entra na lista `resultados`, mostrada na última célula.

In [2]:
resultados = []


def registrar(secao, conferencia, valor, esperado=0, detalhe=""):
    ok = valor == esperado
    resultados.append({
        "seção": secao, "conferência": conferencia, "valor": valor,
        "esperado": esperado, "situação": "ok" if ok else "ATENÇÃO", "detalhe": detalhe,
    })
    return ok


def observar(secao, achado, valor, detalhe=""):
    resultados.append({
        "seção": secao, "conferência": achado, "valor": valor,
        "esperado": "", "situação": "achado", "detalhe": detalhe,
    })


def ler(chave):
    return pd.read_csv(PASTA_DADOS / ARQUIVOS[chave], dtype=str, encoding="utf-8", keep_default_na=False)


dados = {chave: ler(chave) for chave in ARQUIVOS}
catraca, estudantes, componentes = dados["catraca"], dados["estudantes"], dados["componentes"]

S = "1 contagens"
for chave, df in dados.items():
    registrar(S, f"linhas de {ARQUIVOS[chave]}", len(df), LINHAS_ESPERADAS[chave])
registrar(S, "catraca: tentativas servidas", int((catraca["situacao_evento"] == "servida").sum()), SERVIDAS_ESPERADAS)
registrar(S, "catraca: tentativas negadas", int((catraca["situacao_evento"] == "negada").sum()), NEGADAS_ESPERADAS)

momento = pd.to_datetime(catraca["data_hora_refeicao"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
registrar(S, "catraca: datas inválidas", int(momento.isna().sum()))
observar(S, "catraca: cobertura", f"{momento.min()} a {momento.max()}")

# Cópia de trabalho com a data do evento; os dados lidos ficam como vieram.
eventos = catraca.assign(data=momento.dt.normalize().astype("datetime64[ns]"), hora=momento.dt.hour)

## 2. Vazios, chaves e categorias

In [3]:
S = "2 vazios, chaves e categorias"
for chave, df in dados.items():
    vazios = int(df.apply(lambda c: c.str.strip() == "").sum().sum())
    registrar(S, f"{chave}: campos vazios", vazios)
    registrar(S, f"{chave}: chave {CHAVES[chave]} repetida", int(df.duplicated(CHAVES[chave]).sum()))
registrar(S, "catraca: linhas inteiras repetidas", int(catraca.duplicated().sum()))

for chave, regras in CATEGORIAS.items():
    for coluna, permitidos in regras.items():
        valores = dados[chave][coluna]
        fora = ~valores.isin(permitidos)
        registrar(S, f"{chave}.{coluna} fora do dicionário", int(fora.sum()),
                  detalhe=", ".join(sorted(valores[fora].unique())[:5]))

for (chave, coluna), padrao in FORMATOS.items():
    registrar(S, f"{chave}.{coluna} fora do formato {padrao}", int((~dados[chave][coluna].str.fullmatch(padrao)).sum()))

## 3. Ligações entre os arquivos

In [4]:
S = "3 ligações"
registrar(S, "acessos sem vínculo em estudantes", int((~catraca["id_matricula"].isin(estudantes["id_matricula"])).sum()))
vinculos = set(zip(estudantes["id_matricula"], estudantes["periodo_letivo"]))
registrar(S, "inscrições sem vínculo e período", sum(par not in vinculos for par in zip(componentes["id_matricula"], componentes["periodo_letivo"])))
registrar(S, "catraca: id_matricula com mais de uma matrícula", int((catraca.groupby("id_matricula")["matricula"].nunique() > 1).sum()))
registrar(S, "estudantes: id_curso com mais de um curso", int((estudantes.groupby("id_curso")["curso"].nunique() > 1).sum()))
registrar(S, "estudantes: curso com mais de um id_curso", int((estudantes.groupby("curso")["id_curso"].nunique() > 1).sum()))
registrar(S, "estudantes: id_matricula com mais de uma matrícula", int((estudantes.groupby("id_matricula")["matricula"].nunique() > 1).sum()))
registrar(S, "estudantes: matrícula com mais de um id_matricula", int((estudantes.groupby("matricula")["id_matricula"].nunique() > 1).sum()))
registrar(S, "estudantes: período de ingresso depois do período letivo", int((estudantes["periodo_ingresso"] > estudantes["periodo_letivo"]).sum()))

por_vinculo = estudantes.groupby("id_matricula")
for coluna in ["tipo_auxilio", "id_curso", "residente", "periodo_ingresso"]:
    registrar(S, f"estudantes: {coluna} muda para o mesmo vínculo", int((por_vinculo[coluna].nunique() > 1).sum()))

cadastro = estudantes.drop_duplicates("id_matricula")[["id_matricula", "matricula", "id_curso", "curso", "nivel_ensino", "tipo_auxilio"]]
cruzado = catraca.merge(cadastro, on="id_matricula", how="left", suffixes=("", "_cad"))
for coluna in ["matricula", "id_curso", "curso", "nivel_ensino", "tipo_auxilio"]:
    registrar(S, f"catraca.{coluna} diferente do cadastro", int((cruzado[coluna] != cruzado[coluna + "_cad"]).sum()))

observar(S, "id_discente com mais de um id_matricula", int((estudantes.groupby("id_discente")["id_matricula"].nunique() > 1).sum()),
         "previsto no dicionário; os vínculos não são fundidos")
observar(S, "combinações distintas de tipo_acesso e tipo_auxilio", catraca.groupby(["tipo_acesso", "tipo_auxilio"]).ngroups,
         "igual ao número de auxílios quando tipo_acesso repete o auxílio")
display(pd.crosstab(catraca["tipo_acesso"], catraca["tipo_auxilio"]))
display(pd.crosstab(estudantes["residente"], estudantes["tipo_auxilio"]))

tipo_auxilio,auxilio_alimentacao,residencia_estudantil,sem_auxilio
tipo_acesso,,,
parcial,0,0,86809
total,42802,95269,0


tipo_auxilio,auxilio_alimentacao,residencia_estudantil,sem_auxilio
residente,,,
False,504,0,1008
True,0,504,0


## 4. Componentes

In [5]:
S = "4 componentes"
numeros = {c: pd.to_numeric(componentes[c], errors="coerce") for c in ["aulas_previstas", "faltas", "frequencia_percentual", "nota_final"]}
registrar(S, "valores não numéricos", int(sum(v.isna().sum() for v in numeros.values())))
registrar(S, "faltas fora de 0 até aulas previstas", int(((numeros["faltas"] < 0) | (numeros["faltas"] > numeros["aulas_previstas"])).sum()))
frequencia = 100 * (numeros["aulas_previstas"] - numeros["faltas"]) / numeros["aulas_previstas"]
registrar(S, "frequência diferente da fórmula", int(((frequencia - numeros["frequencia_percentual"]).abs() > 0.06).sum()))
registrar(S, "nota fora de 0 a 10", int(((numeros["nota_final"] < 0) | (numeros["nota_final"] > 10)).sum()))

formato_hora = componentes["horario_inicio"].str.fullmatch(r"\d{2}:\d{2}") & componentes["horario_fim"].str.fullmatch(r"\d{2}:\d{2}")
registrar(S, "horários fora de HH:MM", int((~formato_hora).sum()))
registrar(S, "horário de início não anterior ao fim", int((componentes.loc[formato_hora, "horario_inicio"] >= componentes.loc[formato_hora, "horario_fim"]).sum()))
dias = componentes["dias_semana_aula"].str.split("|", regex=False).explode()
registrar(S, "códigos de dia de aula fora de SEG a SEX", int((~dias.isin(DIAS_AULA)).sum()))
registrar(S, "inscrição repetida por vínculo, período e componente", int(componentes.duplicated(["id_matricula", "periodo_letivo", "id_componente"]).sum()))

situacao = componentes.merge(estudantes[["id_matricula", "periodo_letivo", "situacao_vinculo_periodo"]], on=["id_matricula", "periodo_letivo"], how="left")
registrar(S, "inscrições de vínculo não ativo", int((situacao["situacao_vinculo_periodo"] != "ativo").sum()))
observar(S, "inscrições por vínculo e período", str(componentes.groupby(["id_matricula", "periodo_letivo"]).size().value_counts().to_dict()))
display(pd.DataFrame({"resultado": componentes["resultado_final"], "nota": numeros["nota_final"], "frequência": numeros["frequencia_percentual"]})
        .groupby("resultado").agg(["min", "max"]).round(1))

nota       frequência       
           min   max        min    max
resultado                             
aprovado   6.0  10.0       75.0  100.0
reprovado  4.0  10.0       60.0  100.0

## 5. Calendário e fases

A fase de cada evento vem do calendário de `estudantes`: usa o início do recesso mais recente anterior à data do evento. Os dias entre o fim de um período letivo e o recesso seguinte ficam como "fora de período".

In [6]:
S = "5 calendário e eventos"
calendario_completo = estudantes.copy()
for coluna in CAMPOS_CALENDARIO:
    calendario_completo[coluna] = pd.to_datetime(calendario_completo[coluna], format="%Y-%m-%d").astype("datetime64[ns]")

registrar(S, "períodos com calendário diferente entre vínculos",
          int((calendario_completo.groupby("periodo_letivo")[CAMPOS_CALENDARIO].nunique() > 1).any(axis=1).sum()))
c = calendario_completo
ordem_ok = ((c["inicio_recesso"] <= c["fim_recesso"]) & (c["fim_recesso"] < c["inicio_periodo_letivo"])
            & (c["inicio_periodo_letivo"] < c["fim_periodo_letivo"])
            & (c["datas_suspensao_atividades"] >= c["inicio_periodo_letivo"])
            & (c["datas_suspensao_atividades"] <= c["fim_periodo_letivo"]))
registrar(S, "linhas com datas do calendário fora de ordem", int((~ordem_ok).sum()))

calendario = (calendario_completo.drop_duplicates("periodo_letivo")[["periodo_letivo"] + CAMPOS_CALENDARIO]
              .sort_values("inicio_recesso").reset_index(drop=True))
base = (eventos[["id_registro_utilizacao", "id_matricula", "data", "situacao_evento", "tipo_auxilio"]]
        .rename(columns={"tipo_auxilio": "aux_catraca"}).sort_values("data"))
fases = pd.merge_asof(base, calendario, left_on="data", right_on="inicio_recesso", direction="backward")
fases["fase"] = np.select(
    [fases["inicio_recesso"].isna(), fases["data"] > fases["fim_periodo_letivo"],
     fases["data"] <= fases["fim_recesso"], fases["data"] < fases["inicio_periodo_letivo"]],
    ["sem calendário", "fora de período", "recesso", "entre recesso e início"], default="letivo")
display(pd.crosstab(fases["fase"], fases["situacao_evento"], margins=True))
for fase, n in fases["fase"].value_counts().items():
    observar(S, f"eventos na fase: {fase}", int(n))

registrar(S, "eventos sem calendário", int((fases["fase"] == "sem calendário").sum()))
ligado = fases[fases["fase"] != "sem calendário"].merge(
    estudantes[["id_matricula", "periodo_letivo", "situacao_vinculo_periodo", "tipo_auxilio"]],
    on=["id_matricula", "periodo_letivo"], how="left")
registrar(S, "eventos sem vínculo no período da data", int(ligado["situacao_vinculo_periodo"].isna().sum()))
registrar(S, "eventos de vínculo não ativo", int((ligado["situacao_vinculo_periodo"].notna() & (ligado["situacao_vinculo_periodo"] != "ativo")).sum()))
registrar(S, "auxílio do evento diferente do cadastro do período", int((ligado["aux_catraca"] != ligado["tipo_auxilio"]).sum()))

registrar(S, "eventos em domingo", int((eventos["data"].dt.day_of_week == 6).sum()))
observar(S, "eventos na data de suspensão", int((fases["data"] == fases["datas_suspensao_atividades"]).sum()))
registrar(S, "refeições com mais de um horário", int((eventos.groupby("tipo_refeicao")["hora"].nunique() > 1).sum()))
servidas = eventos[eventos["situacao_evento"] == "servida"]
registrar(S, "mesmo vínculo servido duas vezes na mesma refeição e dia",
          int((servidas.groupby(["id_matricula", "data", "tipo_refeicao"]).size() > 1).sum()))


def volta_a_ativo(sequencia):
    saiu = False
    for situacao_periodo in sequencia:
        if situacao_periodo != "ativo":
            saiu = True
        elif saiu:
            return True
    return False


sequencias = estudantes.sort_values("periodo_letivo").groupby("id_matricula")["situacao_vinculo_periodo"].apply(list)
observar(S, "vínculos que saem de ativo e voltam", int(sequencias.apply(volta_a_ativo).sum()))

situacao_evento,negada,servida,All
fase,,,
fora de período,34,2149,2183
letivo,3047,205823,208870
recesso,199,13628,13827
All,3280,221600,224880


## 6. Base diária de referência

Só as refeições servidas entram no alvo. Domingo não tem evento e fica fora da base, sem ser preenchido com zero.

In [7]:
S = "6 base diária"
diario = servidas.groupby(["data", "tipo_refeicao"]).size().unstack(fill_value=0).reindex(columns=REFEICOES, fill_value=0)
registrar(S, "soma das servidas na base diária", int(diario.to_numpy().sum()), SERVIDAS_ESPERADAS)

todos_os_dias = pd.date_range(eventos["data"].min(), eventos["data"].max(), freq="D")
sem_evento = todos_os_dias[~todos_os_dias.isin(diario.index)]
registrar(S, "dias sem evento que não são domingo", int((sem_evento.day_of_week != 6).sum()))
observar(S, "domingos sem evento", int((sem_evento.day_of_week == 6).sum()))
observar(S, "dias na base diária", len(diario))
for refeicao in REFEICOES:
    dias_zero = [d.date().isoformat() for d in diario.index[diario[refeicao] == 0]]
    observar(S, f"dias sem {refeicao} servido", len(dias_zero), ", ".join(dias_zero[:5]))
display(diario.groupby(diario.index.year).mean().round(1))

tipo_refeicao,cafe_da_manha,almoco,jantar
data,,,
2019,23.2,44.1,21.4
2020,23.1,43.3,20.8
2021,23.3,44.1,21.2
2022,23.3,44.2,21.0
2023,23.4,44.0,20.8
2024,22.8,42.5,20.8
2025,24.9,45.9,22.1
2026,26.9,50.2,23.3


## 7. Partição temporal

Cortes por data, sem sorteio. As datas em `FIM_TREINO` e `FIM_VALIDACAO` são provisórias e dependem do acordo do grupo. O teste só deve ser usado na avaliação final.

In [8]:
S = "7 partição"
diario["conjunto"] = np.select([diario.index <= FIM_TREINO, diario.index <= FIM_VALIDACAO], ["treino", "validacao"], default="teste")

linhas = []
for conjunto in ["treino", "validacao", "teste"]:
    parte = diario[diario["conjunto"] == conjunto]
    linhas.append({"conjunto": conjunto, "inicio": parte.index.min().date(), "fim": parte.index.max().date(),
                   "dias": len(parte), "servidas_por_dia": round(parte[REFEICOES].sum(axis=1).mean(), 1),
                   **{r: round(parte[r].mean(), 1) for r in REFEICOES}})
resumo = pd.DataFrame(linhas)
display(resumo)

limites = diario.reset_index().groupby("conjunto")["data"].agg(["min", "max"]).loc[["treino", "validacao", "teste"]]
registrar(S, "conjuntos fora de ordem ou sobrepostos", int((limites["max"].shift(1).dropna() >= limites["min"].iloc[1:]).sum()))
registrar(S, "conjuntos vazios", int((resumo["dias"] == 0).sum()))
registrar(S, "dias fora de algum conjunto", len(diario) - int(resumo["dias"].sum()))

divididos = 0
for _, periodo in calendario.iterrows():
    dentro = diario.loc[periodo["inicio_periodo_letivo"]:periodo["fim_periodo_letivo"], "conjunto"]
    divididos += int(dentro.nunique() > 1)
registrar(S, "períodos letivos divididos entre conjuntos", divididos)
observar(S, "servidas por dia: teste contra treino", f"{resumo.loc[2, 'servidas_por_dia'] / resumo.loc[0, 'servidas_por_dia'] - 1:+.1%}")

,conjunto,inicio,fim,dias,servidas_por_dia,cafe_da_manha,almoco,jantar
0,treino,2019-01-01,2024-12-31,1879,87.9,23.2,43.7,21.0
1,validacao,2025-01-01,2025-12-31,313,92.9,24.9,45.9,22.1
2,teste,2026-01-01,2026-11-14,273,100.4,26.9,50.2,23.3


## 8. Variáveis e vazamento

## Regra usada

A decisão é estimar, na véspera, quantas refeições serão servidas no dia seguinte, por tipo de refeição, para o RU inteiro. Chamo de dia D o dia previsto. Suponho que a previsão é feita no fim do dia D-1, com tudo o que foi registrado até lá e com o que já estava planejado ou publicado antes. Essa suposição precisa ser confirmada pelo grupo: se a previsão for feita mais cedo, os dados mais recentes deixam de estar disponíveis.

Uma variável é permitida quando existe no fim de D-1. É proibida por tempo quando só passa a existir em D ou depois. É proibida por consequência da decisão quando só existe porque alguém já agiu sobre a previsão.

## Classificação

| Variável | Fonte | Classificação | Justificativa |
|---|---|---|---|
| Dia da semana e data | `data_hora_refeicao` (só a data) | Permitida | O calendário é conhecido com antecedência. |
| Fase do calendário (letivo, recesso, suspensão, fora de período) | `estudantes.csv`: início e fim do período, recesso, data de suspensão | Permitida | O calendário acadêmico é definido antes do período. |
| Refeições servidas nos dias anteriores (defasagens e médias móveis até D-1) | `situacao_evento` | Permitida | Usa apenas o que já aconteceu. A média móvel não pode incluir o dia D. |
| Refeições servidas no próprio dia D | `situacao_evento` | Proibida por tempo | É o próprio alvo. |
| Aulas previstas para o dia D (dias, horários e modalidade) | `matriculas_componentes.csv`: `dias_semana_aula`, `horario_inicio`, `horario_fim`, `modalidade` | Permitida | A grade é programada, não observada, e é conhecida antes. |
| Vínculos ativos por tipo de auxílio e residência | `estudantes.csv`: `tipo_auxilio`, `residente` | Permitida com ressalva | O cadastro do período existe antes dos acessos. A ressalva é a linha seguinte. |
| `situacao_vinculo_periodo` | `estudantes.csv` | Permitida com ressalva | Vale para o período inteiro. Um cancelamento ocorrido no meio do período aparece como se valesse desde o início, o que usaria o futuro. Com dados reais, é preciso saber quando a situação é registrada. |
| `tipo_liberacao` do dia D | `catraca_ru.csv` | Proibida por tempo | Só existe quando o acesso é processado. A proporção de liberações manuais de dias anteriores seria permitida. |
| `tipo_acesso` | `catraca_ru.csv` | Proibida por tempo no dia D, e redundante | Repete `tipo_auxilio` em todos os 224.880 eventos. Se precisar da informação, usar o auxílio dos vínculos. |
| `faltas` e `frequencia_percentual` | `matriculas_componentes.csv` | Proibida por tempo | São consolidadas no fim do período e incluem faltas de dias depois da véspera, até do próprio dia D. |
| `nota_final` e `resultado_final` | `matriculas_componentes.csv` | Proibida por tempo | Só existem quando o período termina. |
| Acessos negados do dia D | `situacao_evento` = `negada` | Proibida por tempo | Só são conhecidos durante o dia D. Se, nos dados reais, a negativa acontecer porque a comida acabou, passa a ser também consequência da quantidade preparada. |
| Quantidade preparada, sobras e desperdício | não existe no mock | Proibida por consequência da decisão | A quantidade preparada é a decisão tomada com base na previsão, e as sobras dependem dela. |

## Três exemplos para o relatório individual

Variável permitida: o dia da semana, ou as refeições servidas até D-1. Ambos são conhecidos no fim da véspera e não carregam nenhuma informação do dia previsto.

Variável proibida por tempo: `frequencia_percentual`. O valor é consolidado no fim do período, então inclui as faltas dos dias que ainda não aconteceram quando a previsão é feita. Um modelo que usasse essa coluna parecia prever bem no treino e não teria o dado na véspera.

Variável proibida por consequência da decisão: a quantidade preparada de refeições. Ela não está no mock, mas é a decisão que a previsão deveria orientar. Usá-la para prever a demanda faria o modelo aprender a própria decisão e esconderia o erro real.

## Ligação com a partição

Os conjuntos de treino, validação e teste são separados por data. Qualquer transformação (médias, escalas, codificações) deve ser ajustada só no treino e aplicada depois à validação e ao teste. As defasagens e médias móveis devem ser calculadas dentro da série ordenada por data, sem olhar para o dia previsto nem para os dias seguintes.

## 9. Resultado das conferências

Linhas com `ATENÇÃO` têm valor diferente do esperado e precisam ser explicadas no relatório de qualidade. Linhas com `achado` são observações sem valor esperado.

In [9]:
tabela = pd.DataFrame(resultados)
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 200)
print("Conferências:", int((tabela["situação"] != "achado").sum()),
      "| ok:", int((tabela["situação"] == "ok").sum()),
      "| ATENÇÃO:", int((tabela["situação"] == "ATENÇÃO").sum()),
      "| achados:", int((tabela["situação"] == "achado").sum()))
display(tabela[tabela["situação"] != "ok"])

Conferências: 71 | ok: 71 | ATENÇÃO: 0 | achados: 15


,seção,conferência,valor,esperado,situação,detalhe
6,1 contagens,catraca: cobertura,2019-01-01 07:05:00 a 2026-11-14 18:58:00,,achado,
48,3 ligações,id_discente com mais de um id_matricula,12,,achado,previsto no dicionário; os vínculos não são fundidos
49,3 ligações,combinações distintas de tipo_acesso e tipo_auxilio,3,,achado,igual ao número de auxílios quando tipo_acesso repete o auxílio
59,4 componentes,inscrições por vínculo e período,{3: 1974},,achado,
62,5 calendário e eventos,eventos na fase: letivo,208870,,achado,
63,5 calendário e eventos,eventos na fase: recesso,13827,,achado,
64,5 calendário e eventos,eventos na fase: fora de período,2183,,achado,
70,5 calendário e eventos,eventos na data de suspensão,469,,achado,
73,5 calendário e eventos,vínculos que saem de ativo e voltam,3,,achado,
76,6 base diária,domingos sem evento,410,,achado,
